<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/sugarcnemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
import pandas as pp
import torchvision
from torchvision import datasets
from torch.utils.data import DataLoader

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!unzip -b /content/drive/MyDrive/sugrcanenewfileimagecolab.zip  -d /content/

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
from pathlib import Path
train = Path('/content/dataset/train')
test = Path('/content/dataset/test')
cal = Path('/content/valdiation')

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT


In [ ]:
from torchvision import transforms

manuly = transforms.Compose([

    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=(-10,10)),
    transforms.ToTensor()
])

In [ ]:
auto_transform = weights.transforms()
auto_transform

In [ ]:
!rm rf /content/dataset/train/healthy

In [ ]:
import os
import random
import shutil

image_folder = "/content/dataset/train/yellow"
# label_folder = "/content/sugarcaneimge/labels"
output_folder = "/content/dataset"

random.seed(42)

images = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.shuffle(images)

split = int(len(images) * 0.8)
train_images = images[:split]
val_images = images[split:]

for folder in [
    "images/train",
    "images/val",
    # "labels/train",
    # "labels/val"
]:
    os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

def copy_files(image_list, split_name):
    for image in image_list:
        name = os.path.splitext(image)[0]

        shutil.copy(
            os.path.join(image_folder, image),
            os.path.join(output_folder, "images", split_name, image)
        )

        # label = name + ".txt"
        # label_path = os.path.join(label_folder, label)

        # if os.path.exists(label_path):
        #     shutil.copy(
        #         label_path,
        #         os.path.join(output_folder, "labels", split_name, label)
        #     )

copy_files(train_images, "train")
# copy_files(val_images, "val")

print("Train:", len(train_images))
# print("Validation:", len(val_images))

In [ ]:
!rm -rf /content/dataset/train/.ipynb_checkpoints

In [ ]:
import shutil
import os
from pathlib import Path

# Define the root paths for train and test datasets
train_root = '/content/dataset/train'
test_root = '/content/dataset/test'

# Directories to remove from train_root if they exist and are problematic/empty
train_dirs_to_clean = ['.ipynb_checkpoints', 'redrot', 'rust']
# Directories to remove from test_root if they exist and are problematic/empty
# The error indicates 'healthy' and 'yellow' are problematic for the test set.
test_dirs_to_clean = ['.ipynb_checkpoints', 'redrot', 'rust', 'healthy', 'yellow']

# Clean train_root
for p_dir in train_dirs_to_clean:
    full_path = os.path.join(train_root, p_dir)
    if os.path.isdir(full_path):
        # Only remove if it doesn't contain any image files, or is completely empty.
        if not any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')) for f in os.listdir(full_path)):
            try:
                shutil.rmtree(full_path)
                print(f"Removed problematic (empty/non-image) directory from train_root: {full_path}")
            except OSError as e:
                print(f"Error removing directory {full_path}: {e}")

# Clean test_root
for p_dir in test_dirs_to_clean:
    full_path = os.path.join(test_root, p_dir)
    if os.path.isdir(full_path):
        if not any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp')) for f in os.listdir(full_path)):
            try:
                shutil.rmtree(full_path)
                print(f"Removed problematic (empty/non-image) directory from test_root: {full_path}")
            except OSError as e:
                print(f"Error removing directory {full_path}: {e}")

# Redefine train and test paths as Path objects.
# This ensures ImageFolder receives clean paths.
train = Path(train_root)
test = Path(test_root)

train = datasets.ImageFolder(
    train,
    transform = manuly
)

test = datasets.ImageFolder(
    test,
    transform= manuly
)

In [ ]:
train_dalaloa = DataLoader(
    train,
    batch_size=32,
    shuffle=True
)

test_dalaloa = DataLoader(
    test,
    batch_size=32,
    shuffle=False
)

In [ ]:
#model
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

In [ ]:
!pip install -q torchinfo
from torchinfo import summary
summary(model,input_size=(32,3,224,224))

In [ ]:
t = train.classes

In [ ]:
from torch.nn import parameter
for params in model.features.parameters():
  params.requires_grad = False



In [ ]:
torch.cuda.manual_seed(42)
out = len(t)
model.classifier= torch.nn.Sequential(
    torch.nn.Dropout(0.2),
    torch.nn.Linear(in_features=1280,out_features=out,bias=True)
).to(device)

In [ ]:
summary(model,input_size=(32,3,224,224))

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(),lr=1e-4)

In [ ]:
epochs=20

# Ensure model is on the correct device if there was a discrepancy
model.to(device)

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_dalaloa:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = loss_fn(outputs, batch_labels)

    # back pass
    optim.zero_grad()
    loss.backward()

    # update grads
    optim.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_dalaloa)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

In [ ]:
model.eval()

In [ ]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_dalaloa:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

In [ ]:
import